In [285]:
import numpy as np

In [299]:
def perform_feature_matching(NCCs_left, NCCs_right):
    best_matching_corner_list = []
    second_best_matching_corner_list = []
    for lx, ly, lncc in NCCs_left:
        best = float('-inf')
        second_best = float('-inf')
        rx_best, ry_best = -1, -1
        rx_second_best, ry_second_best = -1, -1

        for rx, ry, rncc in NCCs_right:
            ncc = (lncc * rncc).sum()

            if ncc > best:
                second_best = best
                rx_second_best = rx_best
                ry_second_best = ry_best

                best = ncc
                rx_best = rx
                ry_best = ry
            
            # elif ncc > second_best: # and (rx, ry) != (rx_best, ry_best):
            #     second_best = ncc
            #     rx_second_best = rx
            #     ry_second_best = ry

        if rx_best != -1:
            best_matching_corner_list.append(
                ((lx, ly), (rx_best, ry_best), best))
            if rx_second_best != -1:
                second_best_matching_corner_list.append(
                    ((lx, ly), (rx_second_best, ry_second_best), second_best))
            else:
                print(f"Warning: No second best match found for left corner ({lx}, {ly})")
    return best_matching_corner_list, second_best_matching_corner_list

In [300]:
prefix = "assets/ncc/ncc_"
NCCs_left_component = np.load(prefix + "left_component.npy", allow_pickle=True)
NCCs_right_component = np.load(prefix + "right_component.npy", allow_pickle=True)


In [301]:
best_matching_corner_list, second_best_matching_corner_list = perform_feature_matching(NCCs_left_component, NCCs_right_component)

In [302]:
prefix = "assets/ncc/"
expected_best_matching = np.load(prefix + "best_matching_corner_list.npy", allow_pickle=True)
expected_second_best_matching = np.load(prefix + "second_best_matching_corner_list.npy", allow_pickle=True)

In [303]:
len(expected_best_matching), len(expected_second_best_matching)

(118, 117)

In [304]:
len(best_matching_corner_list), len(second_best_matching_corner_list)

(118, 117)

In [305]:
expected_best_matching[:5]

array([[(np.int64(11), np.int64(91)), (np.int64(11), np.int64(21)),
        np.float64(1.0)],
       [(np.int64(14), np.int64(92)), (np.int64(14), np.int64(22)),
        np.float64(1.0000000000000002)],
       [(np.int64(19), np.int64(86)), (np.int64(19), np.int64(16)),
        np.float64(1.0000000000000002)],
       [(np.int64(19), np.int64(131)), (np.int64(19), np.int64(61)),
        np.float64(0.9999999999999999)],
       [(np.int64(20), np.int64(140)), (np.int64(20), np.int64(70)),
        np.float64(1.0000000000000002)]], dtype=object)

In [306]:
expected_second_best_matching[:5]

array([[(np.int64(14), np.int64(92)), (np.int64(11), np.int64(21)),
        np.float64(0.14847027130153428)],
       [(np.int64(19), np.int64(86)), (np.int64(11), np.int64(152)),
        np.float64(0.278556496704178)],
       [(np.int64(19), np.int64(131)), (np.int64(11), np.int64(21)),
        np.float64(0.022869000639856467)],
       [(np.int64(20), np.int64(140)), (np.int64(11), np.int64(21)),
        np.float64(0.3920550633189286)],
       [(np.int64(25), np.int64(82)), (np.int64(19), np.int64(16)),
        np.float64(0.5820105431604901)]], dtype=object)

In [307]:
best_matching_corner_list[:5]

[((np.int64(11), np.int64(91)), (np.int64(11), np.int64(21)), np.float64(1.0)),
 ((np.int64(14), np.int64(92)),
  (np.int64(14), np.int64(22)),
  np.float64(1.0000000000000002)),
 ((np.int64(19), np.int64(86)),
  (np.int64(19), np.int64(16)),
  np.float64(1.0000000000000002)),
 ((np.int64(19), np.int64(131)),
  (np.int64(19), np.int64(61)),
  np.float64(0.9999999999999999)),
 ((np.int64(20), np.int64(140)),
  (np.int64(20), np.int64(70)),
  np.float64(1.0000000000000002))]

In [308]:
second_best_matching_corner_list[:5]

[((np.int64(14), np.int64(92)),
  (np.int64(11), np.int64(21)),
  np.float64(0.14847027130153428)),
 ((np.int64(19), np.int64(86)),
  (np.int64(11), np.int64(152)),
  np.float64(0.278556496704178)),
 ((np.int64(19), np.int64(131)),
  (np.int64(11), np.int64(21)),
  np.float64(0.022869000639856467)),
 ((np.int64(20), np.int64(140)),
  (np.int64(11), np.int64(21)),
  np.float64(0.3920550633189286)),
 ((np.int64(25), np.int64(82)),
  (np.int64(19), np.int64(16)),
  np.float64(0.5820105431604901))]

In [ ]:
def matches_to_rows(matches):
    return np.array(
        [[lx, ly, rx, ry, score] for (lx, ly), (rx, ry), score in matches],
        dtype=float,
    )

def save_matches_to_csv(matches, filename):
    rows = matches_to_rows(matches)
    np.savetxt(
        filename,
        rows,
        delimiter=",",
        fmt=["%d", "%d", "%d", "%d", "%.18e"],
    )

save_matches_to_csv(best_matching_corner_list, "tmp/best_matching_corner_list.csv")
save_matches_to_csv(second_best_matching_corner_list, "tmp/second_best_matching_corner_list.csv")
save_matches_to_csv(expected_best_matching, "tmp/expected_best_matching.csv")
save_matches_to_csv(expected_second_best_matching, "tmp/expected_second_best_matching.csv")


In [313]:
# Check element-wise equality handling nested structures
def compare_elements(b, e):
    """Compare two elements that may contain nested arrays/tuples"""
    lx_b, ly_b = b[0]
    rx_b, ry_b = b[1]
    score_b = b[2]
    
    lx_e, ly_e = e[0]
    rx_e, ry_e = e[1]
    score_e = e[2]
    
    coords_match = (lx_b == lx_e and ly_b == ly_e and 
                    rx_b == rx_e and ry_b == ry_e)
    
    # Compare scores with tolerance for floating point
    score_match = np.isclose(score_b, score_e, rtol=1e-15)
    
    return coords_match and score_match

all_best_equal = all(
    compare_elements(b, e) for b, e in zip(best_matching_corner_list, expected_best_matching)
)
print(f"\nAll best elements equal: {all_best_equal}")

all_second_best_equal = all(
    compare_elements(b, e) for b, e in zip(second_best_matching_corner_list, expected_second_best_matching)
)
print(f"\nAll second-best elements equal: {all_second_best_equal}")



All best elements equal: True

All second-best elements equal: True
